In [11]:
import os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DB_PATH = os.path.join(BASE_DIR, "2_DATABASE", "eko_cards.db")
PRICES_PATH = os.path.join(BASE_DIR, "3_PROJECT_DATA", "prices.xlsx")

In [12]:
def merge_tables():
    import sqlite3
    import pandas as pd

    conn = sqlite3.connect(DB_PATH)

    query = """
    SELECT
        t.*,
        c.company,
        p.margin,
        p.discount,
        p.final_price AS base_price,

        CASE
            WHEN COALESCE(p.discount, 0) > 0 THEN
                CAST(
                    (
                        CASE
                            WHEN (t.price - COALESCE(p.discount, 0)) < 0 THEN 0
                            ELSE (t.price - COALESCE(p.discount, 0))
                        END
                    ) * 100 + 0.5
                AS INTEGER) / 100.0

            WHEN p.final_price IS NOT NULL THEN
                CAST(p.final_price * 100 + 0.5 AS INTEGER) / 100.0

            ELSE
                CAST(t.price * 100 + 0.5 AS INTEGER) / 100.0
        END AS final_price

    FROM transactions t

    LEFT JOIN cards c
        ON t.card_number = c.card_number

    LEFT JOIN prices p
        ON p.eik = c.eik
        AND UPPER(p.product) = UPPER(t.material)
        AND p.date = (
            SELECT MAX(p2.date)
            FROM prices p2
            WHERE p2.eik = c.eik
              AND UPPER(p2.product) = UPPER(t.material)
              AND p2.date <= t.date
        )
        AND p.rowid = (
            SELECT p3.rowid
            FROM prices p3
            WHERE p3.eik = c.eik
              AND UPPER(p3.product) = UPPER(t.material)
              AND p3.date = (
                  SELECT MAX(p4.date)
                  FROM prices p4
                  WHERE p4.eik = c.eik
                    AND UPPER(p4.product) = UPPER(t.material)
                    AND p4.date <= t.date
              )
            ORDER BY p3.rowid DESC
            LIMIT 1
        )
    """

    data = pd.read_sql(query, conn)
    conn.close()

    return data

In [13]:
def test_row_count():
    df = merge_tables()

    import sqlite3
    conn = sqlite3.connect(DB_PATH)
    total = conn.execute("SELECT COUNT(*) FROM transactions").fetchone()[0]
    conn.close()

    print("Transactions:", total)
    print("Result rows:", len(df))

    assert len(df) == total, "❌ Има дублиране или липсващи редове"

In [14]:
test_row_count()

Transactions: 1181
Result rows: 1181


In [15]:
def test_show_duplicates():
    df = merge_tables()

    duplicates = df[df.duplicated(subset=['id'], keep=False)]

    print("Duplicate rows:", len(duplicates))
    print("Unique duplicated transactions:", duplicates['id'].nunique())

    return duplicates.sort_values('id')

In [16]:
dup_df = test_show_duplicates()
dup_df.head(20)

Duplicate rows: 0
Unique duplicated transactions: 0


,id,plant,name,card_number,material,date,bill_qty,bill_qty2,price,amount,auth_time,Km_stand,company,margin,discount,base_price,final_price
